In [ ]:
!pip install darts -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from pathlib import Path
from darts import TimeSeries

# ============ CONFIG ============
DATA_DIR = Path("/content/drive/MyDrive/Thesis/Datasets/network_anomaly")
SUBNET_DIR = DATA_DIR / "institution_subnets/agg_1_hour"# folder with 548 subnet CSVs
TIMES_PATH = DATA_DIR / "times/times_1_hour.csv"           # file that maps id_time -> time

FEATURE_COLS = [
    "n_flows","n_packets","n_bytes",
    "sum_n_dest_asn","average_n_dest_asn","std_n_dest_asn",
    "sum_n_dest_ports","average_n_dest_ports","std_n_dest_ports",
    "sum_n_dest_ip","average_n_dest_ip","std_n_dest_ip",
    "tcp_udp_ratio_packets","tcp_udp_ratio_bytes",
    "dir_ratio_packets","dir_ratio_bytes",
    "avg_duration","avg_ttl"
]

# ============ LOAD TIME MAP ============

times_df = pd.read_csv(TIMES_PATH)
times_df["id_time"] = times_df["id_time"].astype(int)
times_df["time"] = pd.to_datetime(times_df["time"])

# ============ BUILD SERIES DICTS ============

raw_series = {}   # subnet_id -> pandas DataFrame indexed by time
ts_dict    = {}   # subnet_id -> Darts TimeSeries

for csv_path in SUBNET_DIR.glob("*.csv"):
    sid = csv_path.stem
    df = pd.read_csv(csv_path)

    # Ensure id_time is int to merge correctly
    df["id_time"] = df["id_time"].astype(int)

    # Merge to get real timestamps
    df = df.merge(times_df, on="id_time", how="left")

    # Sort by time and set as index
    df = df.sort_values("time").set_index("time")

    # Keep only feature columns
    df = df[FEATURE_COLS].copy()

    raw_series[sid] = df

    # Build Darts TimeSeries (hourly)
    ts = TimeSeries.from_dataframe(
        df,
        value_cols=FEATURE_COLS,
        fill_missing_dates=True,
        freq="h"
    )
    ts_dict[sid] = ts

print(f"Loaded {len(ts_dict)} subnets into ts_dict")


In [ ]:
import numpy as np

BASE_OUT=Path("/content/drive/MyDrive/Thesis/New_Results/Clustering")
BASE_OUT.mkdir(parents=True, exist_ok=True)

rows = []

for sid, df in raw_series.items():
    feats = {"subnet": sid}

    x = df["n_bytes"].astype(float)

    feats["bytes_mean"]      = x.mean()
    feats["bytes_std"]       = x.std()
    feats["bytes_cv"]        = x.std() / (x.mean() + 1e-9)
    feats["bytes_p90"]       = x.quantile(0.90)
    feats["bytes_zero_frac"] = (x == 0).mean()

    t = np.arange(len(df))
    y = np.log1p(x)
    slope = np.polyfit(t, y, 1)[0]
    feats["bytes_trend"] = slope

    rows.append(feats)

features_df = pd.DataFrame(rows).set_index("subnet").sort_index()


FEATURES_PATH = BASE_OUT / "subnet_features_for_clustering_raw.csv"
features_df.to_csv(FEATURES_PATH)
print("Saved subnet features to:", FEATURES_PATH)
features_df.head()


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Load features
features_df = pd.read_csv(FEATURES_PATH, index_col="subnet")

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features_df.values)

# Try different k
results = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    results.append((k, sil))

sil_df = pd.DataFrame(results, columns=["k", "silhouette"])
print(sil_df)


In [ ]:
#BEST_K = 2  # adjust after seeing sil_df


sil_df = pd.DataFrame(results, columns=["k", "silhouette"])
print(sil_df)

BEST_K = sil_df.loc[sil_df["silhouette"].idxmax(), "k"]
print("Best k by silhouette:", BEST_K)


kmeans = KMeans(n_clusters=BEST_K, random_state=42, n_init="auto")
cluster_labels = kmeans.fit_predict(X_scaled)

features_df["cluster"] = cluster_labels

CLUSTERS_PATH = BASE_OUT / "subnet_clusters_raw.csv"
features_df.to_csv(CLUSTERS_PATH)
print("Saved clusters to:", CLUSTERS_PATH)
features_df.head()
